In [1]:
import os
import re
import pickle
import unicodedata
import random

import pandas as pd
import numpy as np

BASE_DIR = ".."

TRAIN_DIR = os.path.join(BASE_DIR, "data", "train")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

SOURCE1 = os.path.join(TRAIN_DIR, "train_source1.tsv")
SOURCE2 = os.path.join(TRAIN_DIR, "train_source2.tsv")
SOURCE3 = os.path.join(TRAIN_DIR, "train_source3.tsv")
GROUND_TRUTH = os.path.join(TRAIN_DIR, "train_ground_truth.tsv")

VALIDATION_IDS_FILE = os.path.join(
    OUTPUT_DIR,
    "validation_s1_ids.txt"
)

CAPTURED_PATH = os.path.join(
    OUTPUT_DIR,
    "blocking_threshold_10_captured.pkl"
)

CHUNK_SIZE = 100_000
RANDOM_SEED = 42

In [2]:
with open(VALIDATION_IDS_FILE, "r", encoding="utf-8") as f:
    validation_s1_ids = {
        line.strip()
        for line in f
        if line.strip()
    }

with open(CAPTURED_PATH, "rb") as f:
    captured_true_targets = pickle.load(f)

print("Validation S1:", len(validation_s1_ids))
print("Captured true targets:", len(captured_true_targets))

Validation S1: 220682
Captured true targets: 572131


In [3]:
validation_ground_truth = {}

for chunk in pd.read_csv(
    GROUND_TRUTH,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    selected = chunk[
        chunk["source1_entity_id"].isin(validation_s1_ids)
    ]

    for row in selected.itertuples(index=False):

        value = row.matched_entity_ids.strip()

        if value:
            matches = {
                x.strip()
                for x in value.split(",")
                if x.strip()
            }
        else:
            matches = set()

        validation_ground_truth[
            row.source1_entity_id
        ] = matches

total_true_targets = sum(
    len(x)
    for x in validation_ground_truth.values()
)

print("Validation S1 labels:", len(validation_ground_truth))
print("Total validation true matches:", total_true_targets)

Validation S1 labels: 220682
Total validation true matches: 764045


In [4]:
all_true_targets = set()

for target_ids in validation_ground_truth.values():
    all_true_targets.update(target_ids)

missed_true_targets = (
    all_true_targets - captured_true_targets
)

print("Total true targets:", len(all_true_targets))
print("Captured:", len(captured_true_targets))
print("Missed:", len(missed_true_targets))
print(
    "Missed percentage:",
    len(missed_true_targets) / len(all_true_targets) * 100
)

Total true targets: 764045
Captured: 572131
Missed: 191914
Missed percentage: 25.118154035429853


In [5]:
rng = random.Random(RANDOM_SEED)

sample_size = min(
    1000,
    len(missed_true_targets)
)

sampled_missed_targets = rng.sample(
    list(missed_true_targets),
    sample_size
)

sampled_missed_targets = set(
    sampled_missed_targets
)

print(
    "Sampled missed target IDs:",
    len(sampled_missed_targets)
)

Sampled missed target IDs: 1000


In [6]:
missed_target_to_s1 = {}

for s1_id, target_ids in validation_ground_truth.items():
    for target_id in target_ids:

        if target_id in sampled_missed_targets:
            missed_target_to_s1[target_id] = s1_id

print(
    "Mapped missed targets:",
    len(missed_target_to_s1)
)

Mapped missed targets: 1000


In [7]:
sample_s1_ids = set(
    missed_target_to_s1.values()
)

s1_parts = []

for chunk in pd.read_csv(
    SOURCE1,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    selected = chunk[
        chunk["entity_id"].isin(sample_s1_ids)
    ]

    if len(selected) > 0:
        s1_parts.append(selected)

missed_s1 = pd.concat(
    s1_parts,
    ignore_index=True
)

missed_s1 = missed_s1.rename(
    columns={
        "entity_id": "s1_id",
        "business_name": "s1_name",
        "business_address": "s1_address",
        "country": "s1_country"
    }
)

print("S1 records retrieved:", len(missed_s1))

S1 records retrieved: 992


In [8]:
s2_ids = {
    target_id
    for target_id in sampled_missed_targets
    if target_id.startswith("S2-")
}

s3_ids = {
    target_id
    for target_id in sampled_missed_targets
    if target_id.startswith("S3-")
}

s2_parts = []

for chunk in pd.read_csv(
    SOURCE2,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    selected = chunk[
        chunk["entity_id"].isin(s2_ids)
    ]

    if len(selected) > 0:
        s2_parts.append(selected)

s3_parts = []

for chunk in pd.read_csv(
    SOURCE3,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    selected = chunk[
        chunk["entity_id"].isin(s3_ids)
    ]

    if len(selected) > 0:
        s3_parts.append(selected)

missed_s2 = (
    pd.concat(s2_parts, ignore_index=True)
    if s2_parts
    else pd.DataFrame(
        columns=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    )
)

missed_s3 = (
    pd.concat(s3_parts, ignore_index=True)
    if s3_parts
    else pd.DataFrame(
        columns=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    )
)

print("S2 missed records retrieved:", len(missed_s2))
print("S3 missed records retrieved:", len(missed_s3))

S2 missed records retrieved: 489
S3 missed records retrieved: 511


In [9]:
targets = pd.concat(
    [
        missed_s2.assign(target_source="S2"),
        missed_s3.assign(target_source="S3")
    ],
    ignore_index=True
)

targets = targets.rename(
    columns={
        "entity_id": "target_id",
        "business_name": "target_name",
        "business_address": "target_address",
        "country": "target_country"
    }
)

missed_pairs = (
    pd.DataFrame({
        "target_id": list(missed_target_to_s1.keys()),
        "s1_id": list(missed_target_to_s1.values())
    })
    .merge(
        missed_s1,
        on="s1_id",
        how="left"
    )
    .merge(
        targets,
        on="target_id",
        how="left"
    )
)

print("Missed positive pairs:", len(missed_pairs))

Missed positive pairs: 1000


In [10]:
def normalize_text(value):
    value = "" if value is None else str(value)

    value = unicodedata.normalize(
        "NFKC",
        value
    )

    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
        flags=re.UNICODE
    )

    return value.strip()


def get_tokens(value):
    value = normalize_text(value)

    if not value:
        return set()

    return {
        token
        for token in value.split()
        if len(token) >= 2
    }

In [11]:
missed_pairs["s1_name_norm"] = (
    missed_pairs["s1_name"]
    .map(normalize_text)
)

missed_pairs["target_name_norm"] = (
    missed_pairs["target_name"]
    .map(normalize_text)
)

missed_pairs["s1_address_norm"] = (
    missed_pairs["s1_address"]
    .map(normalize_text)
)

missed_pairs["target_address_norm"] = (
    missed_pairs["target_address"]
    .map(normalize_text)
)

missed_pairs["name_exact"] = (
    missed_pairs["s1_name_norm"]
    ==
    missed_pairs["target_name_norm"]
)

missed_pairs["address_exact"] = (
    missed_pairs["s1_address_norm"]
    ==
    missed_pairs["target_address_norm"]
)

missed_pairs["country_same"] = (
    missed_pairs["s1_country"]
    ==
    missed_pairs["target_country"]
)

In [12]:
missed_pairs["name_shared_tokens"] = [
    len(
        get_tokens(a)
        &
        get_tokens(b)
    )
    for a, b in zip(
        missed_pairs["s1_name"],
        missed_pairs["target_name"]
    )
]

missed_pairs["address_shared_tokens"] = [
    len(
        get_tokens(a)
        &
        get_tokens(b)
    )
    for a, b in zip(
        missed_pairs["s1_address"],
        missed_pairs["target_address"]
    )
]

In [13]:
missed_pairs[
    [
        "name_shared_tokens",
        "address_shared_tokens"
    ]
].describe()

,name_shared_tokens,address_shared_tokens
count,1000.000000,1000.000000
mean,1.809000,4.574000
std,1.384354,2.842288
min,0.000000,0.000000
25%,0.000000,3.000000
50%,2.000000,4.000000
75%,3.000000,6.000000
max,5.000000,18.000000


In [14]:
def categorize_missed_pair(row):

    if (
        row["name_shared_tokens"] == 0
        and row["address_shared_tokens"] == 0
    ):
        return "no_shared_name_or_address_tokens"

    if (
        row["name_shared_tokens"] == 0
        and row["address_shared_tokens"] > 0
    ):
        return "address_only"

    if (
        row["name_shared_tokens"] > 0
        and row["address_shared_tokens"] == 0
    ):
        return "name_only"

    return "name_and_address_shared"
    

missed_pairs["failure_category"] = (
    missed_pairs.apply(
        categorize_missed_pair,
        axis=1
    )
)

failure_summary = (
    missed_pairs["failure_category"]
    .value_counts()
    .to_frame("count")
)

failure_summary["percentage"] = (
    failure_summary["count"]
    / len(missed_pairs)
    * 100
)

failure_summary

,count,percentage
failure_category,,
name_and_address_shared,638,63.8
address_only,284,28.4
name_only,78,7.8


In [20]:
def contains_non_ascii(value):
    return any(
        ord(c) > 127
        for c in str(value)
    )

missed_pairs["s1_name_non_ascii"] = (
    missed_pairs["s1_name"]
    .map(contains_non_ascii)
)

missed_pairs["target_name_non_ascii"] = (
    missed_pairs["target_name"]
    .map(contains_non_ascii)
)

missed_pairs["cross_script_name"] = (
    missed_pairs["s1_name_non_ascii"]
    !=
    missed_pairs["target_name_non_ascii"]
)

print(
    "Country disagreement:",
    (
        missed_pairs["country_same"] == False
    ).sum()
)

print(
    "Potential cross-script names:",
    missed_pairs["cross_script_name"].sum()
)

Country disagreement: 0
Potential cross-script names: 212


In [16]:
display_columns = [
    "s1_id",
    "target_id",
    "target_source",
    "s1_name",
    "target_name",
    "s1_address",
    "target_address",
    "s1_country",
    "target_country",
    "name_shared_tokens",
    "address_shared_tokens",
    "failure_category"
]

missed_pairs[
    display_columns
].head(50)

,s1_id,target_id,target_source,s1_name,target_name,s1_address,target_address,s1_country,target_country,name_shared_tokens,address_shared_tokens,failure_category
0,S1-53064564,S3-157576986,S3,Consultancy Research Ltd,Consultancy-Rseahc Ltd,"No. 158, 8Th Main, 1St Cross, Btm Layout 1St S...","H.no 158/1, Bangalore, 8Th Main, 1St Cross, Bt...",India,India,2,10,name_and_address_shared
1,S1-458279858,S2-313022838,S2,Holly Abacus Inc,Holly Abacus - 4300544811,"117 6th Avenue, Naperville, IL","17 6TH AVE, NAPERVILLE, IL",US,US,2,3,name_and_address_shared
2,S1-117315529,S2-692324089,S2,Systems Black Advertising Limited,limited systems black advertising,"Gat No. 118, P No.12, Tambe Nagar, Tandulwadi,...","GAT NO. 118, BARAMATI, महाराष्ट्र",India,India,4,4,name_and_address_shared
3,S1-810816325,S3-917355077,S3,Aditya Global Industries Private Limited,Aditya Gdloabl Industries Prívate Limited,"3, Pleasant Park, Ravindra High School Road Ho...","Block E-616 3, Pleasant Park, Ravindra High Sc...",India,India,3,11,name_and_address_shared
4,S1-729322951,S3-780073204,S3,Sun Ventures Private Limited,Sun Véntures Private Limited,"Office No 8, 1St Floor, Vidya Society, Off Iti...","Office No #8, Pune, महाराष्ट्र",India,India,3,3,name_and_address_shared
5,S1-522147537,S3-224104624,S3,Continental Grand Hawaii P.C.,Service Continental Grand P.C.,"5858 Happy Valley Road, Tallassee, TN","7858 Happy Valley Road, Tallassee, Tennessee",US,US,2,4,name_and_address_shared
6,S1-554513559,S3-681710047,S3,Foot & Ankle Direct Clinic PLLC,Foot + Ankle Direct Clinic,"Goldsboro, NC, 405 Daw Pate Road",,US,US,4,0,name_only
7,S1-903585562,S2-968513741,S2,Stanley Peak Amc Ltd,Stanley Amc Ltd Center,"Albemarle County, VA, 660 Woodburn Court","ALBEMARLE COUNTY, WOODBURN CT, VA",US,US,3,4,name_and_address_shared
8,S1-20558015,S2-382011466,S2,AN Rocky Chimera Center,AN Rocky Chimera Centre,"890 Trace Drive, Buffalo Grove, IL","BUFFALO GROVE, NULL, IL, TRACE DR",US,US,3,4,name_and_address_shared
9,S1-838975645,S3-16699630,S3,Developers Vijay International Private Limited,Deve1opers Vijay International Private Limited,"Tower No.3, Homeland Height, Sector 70, S.A.S....","Flat No.12, Tower No.3, Homeland Height, Secto...",India,India,4,10,name_and_address_shared


In [17]:
print("Potential cross-script names:", missed_pairs["cross_script_name"].sum())

Potential cross-script names: 212


In [23]:
from collections import Counter
import re
import unicodedata
import pandas as pd
import numpy as np


def normalize_text(value):
    value = "" if value is None else str(value)

    value = unicodedata.normalize("NFKC", value)
    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
        flags=re.UNICODE
    )

    return value.strip()


def get_tokens(value):
    value = normalize_text(value)

    if not value:
        return set()

    return {
        token
        for token in value.split()
        if len(token) >= 2
    }


validation_s1_path = SOURCE1

name_token_frequency = Counter()
address_token_frequency = Counter()

for chunk in pd.read_csv(
    validation_s1_path,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):

    chunk = chunk[
        chunk["entity_id"].isin(validation_s1_ids)
    ]

    for row in chunk.itertuples(index=False):

        name_token_frequency.update(
            get_tokens(row.business_name)
        )

        address_token_frequency.update(
            get_tokens(row.business_address)
        )

print(
    "Unique validation-S1 name tokens:",
    len(name_token_frequency)
)

print(
    "Unique validation-S1 address tokens:",
    len(address_token_frequency)
)

Unique validation-S1 name tokens: 48652
Unique validation-S1 address tokens: 118619


In [24]:
print(
    "Top name tokens:",
    name_token_frequency.most_common(30)
)

print(
    "Top address tokens:",
    address_token_frequency.most_common(30)
)

Top name tokens: [('limited', 52308), ('private', 43192), ('llc', 35516), ('inc', 23722), ('ltd', 14974), ('pvt', 12246), ('india', 5971), ('and', 5856), ('care', 5221), ('of', 4328), ('associates', 4277), ('group', 4009), ('llp', 3948), ('partners', 3561), ('center', 3455), ('corp', 3422), ('services', 3061), ('health', 2857), ('pc', 2759), ('co', 2755), ('clinic', 2716), ('solutions', 2627), ('global', 2528), ('trading', 2265), ('technologies', 2104), ('international', 2039), ('brothers', 2019), ('pllc', 2019), ('industries', 1913), ('medicine', 1846)]
Top address tokens: [('road', 45852), ('no', 36025), ('street', 31080), ('drive', 22288), ('maharashtra', 19232), ('avenue', 19106), ('unit', 19020), ('floor', 16306), ('nagar', 15610), ('city', 14623), ('west', 13357), ('tx', 13179), ('delhi', 12180), ('pradesh', 11329), ('new', 11086), ('lane', 10699), ('ny', 10239), ('nc', 9475), ('mumbai', 8730), ('plot', 8620), ('oh', 8562), ('il', 7588), ('uttar', 7370), ('karnataka', 7005), ('ta

In [25]:
def shared_token_frequencies(tokens_a, tokens_b, frequency_map):
    shared = get_tokens(tokens_a) & get_tokens(tokens_b)

    frequencies = [
        frequency_map[token]
        for token in shared
        if token in frequency_map
    ]

    return frequencies

In [26]:
missed_pairs["shared_name_token_min_freq"] = [
    min(
        shared_token_frequencies(
            a,
            b,
            name_token_frequency
        ),
        default=np.inf
    )
    for a, b in zip(
        missed_pairs["s1_name"],
        missed_pairs["target_name"]
    )
]

missed_pairs["shared_address_token_min_freq"] = [
    min(
        shared_token_frequencies(
            a,
            b,
            address_token_frequency
        ),
        default=np.inf
    )
    for a, b in zip(
        missed_pairs["s1_address"],
        missed_pairs["target_address"]
    )
]

In [27]:
freq_thresholds = [10, 20, 50, 100, 500, 1000]

recovery_analysis = []

for threshold in freq_thresholds:

    name_recoverable = (
        missed_pairs["shared_name_token_min_freq"]
        <= threshold
    )

    address_recoverable = (
        missed_pairs["shared_address_token_min_freq"]
        <= threshold
    )

    either_recoverable = (
        name_recoverable
        |
        address_recoverable
    )

    recovery_analysis.append({
        "threshold": threshold,
        "name_recoverable": name_recoverable.sum(),
        "address_recoverable": address_recoverable.sum(),
        "either_recoverable": either_recoverable.sum(),
        "percentage_of_sample_recoverable": (
            either_recoverable.mean() * 100
        )
    })

recovery_df = pd.DataFrame(
    recovery_analysis
)

recovery_df

,threshold,name_recoverable,address_recoverable,either_recoverable,percentage_of_sample_recoverable
0,10,0,0,0,0.0
1,20,63,255,306,30.6
2,50,167,539,622,62.2
3,100,258,686,755,75.5
4,500,478,854,928,92.8
5,1000,650,878,969,96.9


In [28]:
from itertools import combinations


def get_token_pairs(value):
    tokens = sorted(get_tokens(value))

    if len(tokens) < 2:
        return set()

    return set(combinations(tokens, 2))

In [29]:
missed_pairs["name_token_pairs"] = (
    missed_pairs["s1_name"]
    .map(get_token_pairs)
)

missed_pairs["target_name_token_pairs"] = (
    missed_pairs["target_name"]
    .map(get_token_pairs)
)

missed_pairs["shared_name_pairs"] = [
    set(a) & set(b)
    for a, b in zip(
        missed_pairs["name_token_pairs"],
        missed_pairs["target_name_token_pairs"]
    )
]

In [30]:
missed_pairs["address_token_pairs"] = (
    missed_pairs["s1_address"]
    .map(get_token_pairs)
)

missed_pairs["target_address_token_pairs"] = (
    missed_pairs["target_address"]
    .map(get_token_pairs)
)

missed_pairs["shared_address_pairs"] = [
    set(a) & set(b)
    for a, b in zip(
        missed_pairs["address_token_pairs"],
        missed_pairs["target_address_token_pairs"]
    )
]

In [31]:
pair_recovery_summary = pd.DataFrame({
    "signal": [
        "shared_name_token_pair",
        "shared_address_token_pair",
        "shared_pair_either_name_or_address"
    ],
    "count": [
        (
            missed_pairs["shared_name_pairs"]
            .map(len) > 0
        ).sum(),

        (
            missed_pairs["shared_address_pairs"]
            .map(len) > 0
        ).sum(),

        (
            (
                missed_pairs["shared_name_pairs"].map(len)
                +
                missed_pairs["shared_address_pairs"].map(len)
            ) > 0
        ).sum()
    ]
})

pair_recovery_summary["percentage"] = (
    pair_recovery_summary["count"]
    / len(missed_pairs)
    * 100
)

pair_recovery_summary

,signal,count,percentage
0,shared_name_token_pair,624,62.4
1,shared_address_token_pair,908,90.8
2,shared_pair_either_name_or_address,988,98.8


In [32]:
pair_examples = []

for _, row in missed_pairs.iterrows():

    for pair in list(row["shared_name_pairs"])[:5]:
        pair_examples.append({
            "field": "name",
            "token_1": pair[0],
            "token_2": pair[1]
        })

    for pair in list(row["shared_address_pairs"])[:5]:
        pair_examples.append({
            "field": "address",
            "token_1": pair[0],
            "token_2": pair[1]
        })

pair_examples_df = pd.DataFrame(pair_examples)

pair_examples_df.head(100)

,field,token_1,token_2
0,name,consultancy,ltd
1,address,8th,no
2,address,bangalore,layout
3,address,layout,no
4,address,1st,cross
...,...,...,...
95,address,city,conway
96,address,city,va
97,address,conway,james
98,address,county,va


In [33]:
from collections import Counter
from itertools import combinations

name_pair_frequency = Counter()
address_pair_frequency = Counter()

for chunk in pd.read_csv(
    SOURCE1,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    chunk = chunk[
        chunk["entity_id"].isin(validation_s1_ids)
    ]

    for row in chunk.itertuples(index=False):

        name_tokens = sorted(
            get_tokens(row.business_name)
        )

        address_tokens = sorted(
            get_tokens(row.business_address)
        )

        if len(name_tokens) >= 2:
            name_pair_frequency.update(
                combinations(name_tokens, 2)
            )

        if len(address_tokens) >= 2:
            address_pair_frequency.update(
                combinations(address_tokens, 2)
            )

print(
    "Unique name token pairs:",
    len(name_pair_frequency)
)

print(
    "Unique address token pairs:",
    len(address_pair_frequency)
)

Unique name token pairs: 385258
Unique address token pairs: 2472984


In [34]:
print(
    "Top name pairs:"
)

print(
    name_pair_frequency.most_common(30)
)

print(
    "\nTop address pairs:"
)

print(
    address_pair_frequency.most_common(30)
)

Top name pairs:
[(('limited', 'private'), 43134), (('ltd', 'pvt'), 12246), (('india', 'limited'), 3557), (('india', 'private'), 2912), (('and', 'llc'), 1870), (('limited', 'services'), 1511), (('limited', 'solutions'), 1353), (('and', 'inc'), 1261), (('private', 'services'), 1260), (('limited', 'trading'), 1246), (('india', 'ltd'), 1193), (('brothers', 'limited'), 1132), (('limited', 'technologies'), 1113), (('private', 'solutions'), 1102), (('associates', 'care'), 1068), (('private', 'trading'), 1013), (('care', 'llc'), 942), (('global', 'limited'), 935), (('private', 'technologies'), 931), (('brothers', 'private'), 923), (('limited', 'tech'), 920), (('india', 'pvt'), 920), (('international', 'limited'), 902), (('associates', 'llc'), 815), (('industries', 'limited'), 800), (('agro', 'limited'), 772), (('private', 'tech'), 767), (('global', 'private'), 763), (('developers', 'limited'), 759), (('enterprises', 'limited'), 748)]

Top address pairs:
[(('maharashtra', 'no'), 8872), (('mahar

In [35]:
def shared_pair_frequencies(
    value_a,
    value_b,
    frequency_map
):
    pairs_a = set(
        combinations(
            sorted(get_tokens(value_a)),
            2
        )
    )

    pairs_b = set(
        combinations(
            sorted(get_tokens(value_b)),
            2
        )
    )

    shared_pairs = pairs_a & pairs_b

    frequencies = [
        frequency_map[pair]
        for pair in shared_pairs
        if pair in frequency_map
    ]

    return frequencies

In [36]:
missed_pairs["min_shared_name_pair_freq"] = [
    min(
        shared_pair_frequencies(
            a,
            b,
            name_pair_frequency
        ),
        default=np.inf
    )
    for a, b in zip(
        missed_pairs["s1_name"],
        missed_pairs["target_name"]
    )
]

missed_pairs["min_shared_address_pair_freq"] = [
    min(
        shared_pair_frequencies(
            a,
            b,
            address_pair_frequency
        ),
        default=np.inf
    )
    for a, b in zip(
        missed_pairs["s1_address"],
        missed_pairs["target_address"]
    )
]

In [37]:
pair_frequency_thresholds = [
    2,
    5,
    10,
    20,
    50,
    100,
    500
]

pair_frequency_recovery = []

for threshold in pair_frequency_thresholds:

    name_recoverable = (
        missed_pairs["min_shared_name_pair_freq"]
        <= threshold
    )

    address_recoverable = (
        missed_pairs["min_shared_address_pair_freq"]
        <= threshold
    )

    either_recoverable = (
        name_recoverable
        |
        address_recoverable
    )

    pair_frequency_recovery.append({
        "threshold": threshold,
        "name_recoverable": int(
            name_recoverable.sum()
        ),
        "address_recoverable": int(
            address_recoverable.sum()
        ),
        "either_recoverable": int(
            either_recoverable.sum()
        ),
        "percentage_recoverable": (
            either_recoverable.mean()
            * 100
        )
    })

pair_frequency_recovery_df = pd.DataFrame(
    pair_frequency_recovery
)

pair_frequency_recovery_df

,threshold,name_recoverable,address_recoverable,either_recoverable,percentage_recoverable
0,2,282,756,812,81.2
1,5,390,797,864,86.4
2,10,476,815,900,90.0
3,20,529,837,926,92.6
4,50,553,869,950,95.0
5,100,566,878,958,95.8
6,500,616,891,977,97.7
